In [2]:
# ============================================
# 01_data_explore_manifest_and_params.ipynb
# Section 1. Imports, paths, constants
# (Spontaneous EEG only)
# ============================================

from __future__ import annotations

from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

# Optional: only needed later for light loading/QC
try:
    import mne
except Exception:
    mne = None

# ----------------------------
# Project paths
# ----------------------------
PROJECT_ROOT = Path("..").resolve()

DATA_ROOT = PROJECT_ROOT / "data"
SPONT_PATH = DATA_ROOT / "raw"/ "spontaneous"

DERIVED_ROOT = DATA_ROOT / "derived"
MANIFEST_DIR = DERIVED_ROOT / "manifests"
QC_DIR = DERIVED_ROOT / "qc"
CONFIG_DIR = DERIVED_ROOT / "config"

for d in [MANIFEST_DIR, QC_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPONT_PATH:", SPONT_PATH)
print("DERIVED_ROOT:", DERIVED_ROOT)

# ----------------------------
# Expected structure
# ----------------------------
EXPECTED_FILES_PER_SUBJECT = 4
EXPECTED_EC_PER_SUBJECT = 2
EXPECTED_EO_PER_SUBJECT = 2


PROJECT_ROOT: /Users/I743312/Documents/ketamine project
SPONT_PATH: /Users/I743312/Documents/ketamine project/data/raw/spontaneous
DERIVED_ROOT: /Users/I743312/Documents/ketamine project/data/derived


In [3]:
# ============================================
# Section 2. File discovery (spontaneous only)
# ============================================

def discover_spontaneous_set_files(spont_path: Path) -> pd.DataFrame:
    """
    Discover EEGLAB .set files in the spontaneous folder.
    Ignores .fdt files by design.
    """
    if not spont_path.exists():
        raise FileNotFoundError(f"Spontaneous path not found: {spont_path}")

    set_files = sorted(spont_path.rglob("*.set"))

    rows = []
    for p in set_files:
        rows.append({
            "file_path": str(p),
            "file_name": p.name,
            "parent_dir": str(p.parent),
        })

    return pd.DataFrame(rows)

df_files = discover_spontaneous_set_files(SPONT_PATH)

print("Total spontaneous .set files found:", len(df_files))
print("Example files:")
df_files.head(10)


Total spontaneous .set files found: 40
Example files:


,file_path,file_name,parent_dir
0,/Users/I743312/Documents/ketamine project/data...,210_20161207_0002_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
1,/Users/I743312/Documents/ketamine project/data...,210_20161207_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...
2,/Users/I743312/Documents/ketamine project/data...,210_20161207_0006_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
3,/Users/I743312/Documents/ketamine project/data...,210_20161207_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...
4,/Users/I743312/Documents/ketamine project/data...,219_20161117_0002_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
5,/Users/I743312/Documents/ketamine project/data...,219_20161117_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...
6,/Users/I743312/Documents/ketamine project/data...,219_20161117_0006_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
7,/Users/I743312/Documents/ketamine project/data...,219_20161117_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...
8,/Users/I743312/Documents/ketamine project/data...,249_20161208_0002_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
9,/Users/I743312/Documents/ketamine project/data...,249_20161208_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...


In [4]:
# ============================================
# Section 3. Strict filename parsing
# and initial manifest creation (spontaneous)
# ============================================

# Expected filename pattern (normalized):
# <subject>_<YYYYMMDD>_<4digitRec>_<eyesOpen|eyesClosed>_afterICA.set

FNAME_REGEX = re.compile(
    r"^(?P<subject>\d+)_"
    r"(?P<date>\d{8})_"
    r"(?P<rec>\d{4})_"
    r"(?P<eyes>eyesOpen|eyesClosed)_"
    r"afterICA\.set$",
    flags=re.IGNORECASE
)

def parse_from_filename(file_name: str) -> dict:
    """
    Strict parser for normalized spontaneous EEG filenames.

    Returns a dict with parsed fields and a parse_ok flag.
    Files that do not match exactly are marked parse_ok=False
    and should be excluded from the validated manifest.
    """
    m = FNAME_REGEX.match(file_name)
    if not m:
        return {
            "subject_id": "unknown",
            "date_str": None,
            "recording_number": None,
            "eyes": "unknown",
            "parse_ok": False,
            "parse_notes": "regex_failed",
        }

    eyes_raw = m.group("eyes").lower()
    eyes = "open" if eyes_raw == "eyesopen" else "closed"

    return {
        "subject_id": m.group("subject"),
        "date_str": m.group("date"),
        "recording_number": int(m.group("rec")),
        "eyes": eyes,
        "parse_ok": True,
        "parse_notes": "",
    }

def build_initial_manifest(df_files: pd.DataFrame) -> pd.DataFrame:
    """
    Build the initial manifest by parsing filenames only.
    No drug labels, no QC yet.
    """
    rows = []
    for _, r in df_files.iterrows():
        parsed = parse_from_filename(r["file_name"])
        rows.append({
            **parsed,
            "file_path": r["file_path"],
            "file_name": r["file_name"],
            "parent_dir": r["parent_dir"],
        })
    return pd.DataFrame(rows)

# Build manifest
manifest = build_initial_manifest(df_files)

print("Files before parsing filter:", len(df_files))
print("Files matching strict pattern:", manifest["parse_ok"].sum())

# Drop files that failed strict parsing
manifest = manifest[manifest["parse_ok"]].reset_index(drop=True)

print("Rows in manifest after filtering:", len(manifest))
print("Unique subjects:", manifest["subject_id"].nunique())

# Sanity check: inspect one subject
example_sid = manifest["subject_id"].iloc[0]
manifest[manifest["subject_id"] == example_sid].sort_values("recording_number")


Files before parsing filter: 40
Files matching strict pattern: 40
Rows in manifest after filtering: 40
Unique subjects: 10


,subject_id,date_str,recording_number,eyes,parse_ok,parse_notes,file_path,file_name,parent_dir
0,210,20161207,2,open,True,,/Users/I743312/Documents/ketamine project/data...,210_20161207_0002_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
1,210,20161207,3,closed,True,,/Users/I743312/Documents/ketamine project/data...,210_20161207_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...
2,210,20161207,6,open,True,,/Users/I743312/Documents/ketamine project/data...,210_20161207_0006_eyesOpen_afterICA.set,/Users/I743312/Documents/ketamine project/data...
3,210,20161207,7,closed,True,,/Users/I743312/Documents/ketamine project/data...,210_20161207_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...


In [5]:
# ============================================
# Section 4. Drug assignment (order rule)
# and basic per-subject validation
# ============================================

# ----------------------------
# 4.1 Assign drug label by recording order
# Rule: first two recordings = awake, last two = ketamine
# ----------------------------

def assign_drug_by_order_rule(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign drug labels within each subject based on recording_number order.
    Assumes exactly 4 recordings per subject in the spontaneous dataset.
    """
    df = df.copy()
    df["drug"] = "unknown"
    df["drug_source"] = "order_rule"
    df["drug_order_confidence"] = True

    for sid, g in df.groupby("subject_id", sort=False):
        # Sort strictly by recording number
        g_sorted = g.sort_values("recording_number")
        idx = g_sorted.index.tolist()

        if len(idx) != EXPECTED_FILES_PER_SUBJECT:
            # Leave drug as unknown for this subject
            df.loc[idx, "drug_order_confidence"] = False
            continue

        # Apply rule
        df.loc[idx[:2], "drug"] = "awake"
        df.loc[idx[2:], "drug"] = "ketamine"

    return df

manifest = assign_drug_by_order_rule(manifest)

# Sanity check: inspect one subject
example_sid = manifest["subject_id"].iloc[0]
manifest[manifest["subject_id"] == example_sid][
    ["file_name", "recording_number", "eyes", "drug"]
].sort_values("recording_number")


,file_name,recording_number,eyes,drug
0,210_20161207_0002_eyesOpen_afterICA.set,2,open,awake
1,210_20161207_0003_eyesClosed_afterICA.set,3,closed,awake
2,210_20161207_0006_eyesOpen_afterICA.set,6,open,ketamine
3,210_20161207_0007_eyesClosed_afterICA.set,7,closed,ketamine


In [6]:
# ----------------------------
# 4.2 Per-subject validation for minimal EC-only analysis
# ----------------------------

def validate_subject_group(g: pd.DataFrame) -> dict:
    notes = []

    n_files = len(g)
    if n_files != EXPECTED_FILES_PER_SUBJECT:
        notes.append(f"expected_{EXPECTED_FILES_PER_SUBJECT}_files_got_{n_files}")

    n_ec = (g["eyes"] == "closed").sum()
    n_eo = (g["eyes"] == "open").sum()

    if n_ec != EXPECTED_EC_PER_SUBJECT or n_eo != EXPECTED_EO_PER_SUBJECT:
        notes.append(f"eyes_counts_ec_{n_ec}_eo_{n_eo}")

    has_awake_ec = ((g["eyes"] == "closed") & (g["drug"] == "awake")).any()
    has_ket_ec = ((g["eyes"] == "closed") & (g["drug"] == "ketamine")).any()

    if not (has_awake_ec and has_ket_ec):
        notes.append("missing_ec_for_one_or_both_drug_states")

    if not g["drug_order_confidence"].all():
        notes.append("drug_order_confidence_false")

    return {
        "subject_id": g["subject_id"].iloc[0],
        "passes_basic_checks": len(notes) == 0,
        "check_notes": ";".join(notes),
        "n_files": n_files,
        "n_ec": int(n_ec),
        "n_eo": int(n_eo),
        "has_awake_ec": bool(has_awake_ec),
        "has_ket_ec": bool(has_ket_ec),
    }

qc_rows = []
for sid, g in manifest.groupby("subject_id", sort=False):
    qc_rows.append(validate_subject_group(g))

qc_report = pd.DataFrame(qc_rows)

print("Total subjects:", qc_report["subject_id"].nunique())
print("Subjects passing basic checks:", qc_report["passes_basic_checks"].sum())

# Show failures explicitly
qc_report[~qc_report["passes_basic_checks"]].head(20)


Total subjects: 10
Subjects passing basic checks: 10


,subject_id,passes_basic_checks,check_notes,n_files,n_ec,n_eo,has_awake_ec,has_ket_ec


In [7]:
# ----------------------------
# 4.3 Create validated manifest
# ----------------------------

manifest = manifest.merge(
    qc_report[["subject_id", "passes_basic_checks", "check_notes"]],
    on="subject_id",
    how="left",
)

manifest_validated = manifest[manifest["passes_basic_checks"]].copy()

print("Rows in validated manifest:", len(manifest_validated))
print("Validated subjects:", manifest_validated["subject_id"].nunique())

# Final sanity check: EC-only rows that will be used downstream
manifest_validated_ec = manifest_validated[manifest_validated["eyes"] == "closed"]

print("EC-only rows:", len(manifest_validated_ec))
manifest_validated_ec.head(10)


Rows in validated manifest: 40
Validated subjects: 10
EC-only rows: 20


,subject_id,date_str,recording_number,eyes,parse_ok,parse_notes,file_path,file_name,parent_dir,drug,drug_source,drug_order_confidence,passes_basic_checks,check_notes
1,210,20161207,3,closed,True,,/Users/I743312/Documents/ketamine project/data...,210_20161207_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,
3,210,20161207,7,closed,True,,/Users/I743312/Documents/ketamine project/data...,210_20161207_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,
5,219,20161117,3,closed,True,,/Users/I743312/Documents/ketamine project/data...,219_20161117_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,
7,219,20161117,7,closed,True,,/Users/I743312/Documents/ketamine project/data...,219_20161117_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,
9,249,20161208,3,closed,True,,/Users/I743312/Documents/ketamine project/data...,249_20161208_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,
11,249,20161208,7,closed,True,,/Users/I743312/Documents/ketamine project/data...,249_20161208_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,
13,251,20170124,3,closed,True,,/Users/I743312/Documents/ketamine project/data...,251_20170124_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,
15,251,20170124,7,closed,True,,/Users/I743312/Documents/ketamine project/data...,251_20170124_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,
17,265,20170112,4,closed,True,,/Users/I743312/Documents/ketamine project/data...,265_20170112_0004_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,
19,265,20170112,7,closed,True,,/Users/I743312/Documents/ketamine project/data...,265_20170112_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,


In [12]:
# ============================================
# Section 4.4 Save manifest artifacts (CSV)
# ============================================

MANIFEST_ALL_PATH = MANIFEST_DIR / "manifest_spontaneous_all.csv"
MANIFEST_VALIDATED_PATH = MANIFEST_DIR / "manifest_spontaneous_validated.csv"

manifest.to_csv(MANIFEST_ALL_PATH, index=False)
manifest_validated.to_csv(MANIFEST_VALIDATED_PATH, index=False)

print("Saved manifest files:")
print(" -", MANIFEST_ALL_PATH)
print(" -", MANIFEST_VALIDATED_PATH)

print("Summary:")
print("  total files:", len(manifest))
print("  validated files:", len(manifest_validated))
print("  validated subjects:", manifest_validated["subject_id"].nunique())



Saved manifest files:
 - /Users/I743312/Documents/ketamine project/data/derived/manifests/manifest_spontaneous_all.csv
 - /Users/I743312/Documents/ketamine project/data/derived/manifests/manifest_spontaneous_validated.csv
Summary:
  total files: 40
  validated files: 40
  validated subjects: 10


In [8]:
# ============================================
# Section 5. Data exploration (structural / technical only)
# Spontaneous EEG – validated subjects
# ============================================

import hashlib
from collections import Counter

assert mne is not None, "mne is required for exploration (read_raw_eeglab)"

# ------------------------------------------------
# 5.1 Select ONE file per subject (EC preferred)
# ------------------------------------------------

# We explore EC only, because that is the minimal analysis subset
manifest_ec = manifest_validated[manifest_validated["eyes"] == "closed"].copy()

# Take the first EC file per subject (deterministic)
explore_df = (
    manifest_ec
    .sort_values(["subject_id", "recording_number"])
    .groupby("subject_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Subjects explored:", explore_df["subject_id"].nunique())
print("Files explored:", len(explore_df))

Subjects explored: 10
Files explored: 10


In [9]:
# ------------------------------------------------
# 5.2 Load files and extract structural metadata
# Handles both continuous Raw and epoched Epochs
# ------------------------------------------------

import hashlib

def hash_channel_names(ch_names):
    joined = "|".join(ch_names)
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:12]

rows = []

for _, r in explore_df.iterrows():
    fp = r["file_path"]
    sid = r["subject_id"]

    try:
        # Try continuous first
        raw = mne.io.read_raw_eeglab(fp, preload=False, verbose="ERROR")

        sfreq = float(raw.info["sfreq"])
        n_channels = int(raw.info["nchan"])
        duration_sec = raw.n_times / sfreq
        ch_hash = hash_channel_names(raw.ch_names)

        rows.append({
            "subject_id": sid,
            "file_path": fp,
            "data_type": "raw",
            "sfreq": sfreq,
            "n_channels": n_channels,
            "duration_sec": duration_sec,
            "n_epochs": None,
            "epoch_len_sec": None,
            "channel_hash": ch_hash,
            "load_ok": True,
        })

    except Exception as e_raw:
        # Fallback to epoched data
        try:
            epochs = mne.io.read_epochs_eeglab(fp, verbose="ERROR")

            sfreq = float(epochs.info["sfreq"])
            n_channels = int(epochs.info["nchan"])
            n_epochs = int(len(epochs))
            n_times = int(epochs.get_data().shape[-1])
            epoch_len_sec = n_times / sfreq
            duration_sec = n_epochs * epoch_len_sec  # total samples across epochs
            ch_hash = hash_channel_names(epochs.ch_names)

            rows.append({
                "subject_id": sid,
                "file_path": fp,
                "data_type": "epochs",
                "sfreq": sfreq,
                "n_channels": n_channels,
                "duration_sec": duration_sec,
                "n_epochs": n_epochs,
                "epoch_len_sec": epoch_len_sec,
                "channel_hash": ch_hash,
                "load_ok": True,
                "load_error_raw": str(e_raw),
            })

        except Exception as e_epochs:
            rows.append({
                "subject_id": sid,
                "file_path": fp,
                "data_type": "unreadable",
                "sfreq": None,
                "n_channels": None,
                "duration_sec": None,
                "n_epochs": None,
                "epoch_len_sec": None,
                "channel_hash": None,
                "load_ok": False,
                "load_error_raw": str(e_raw),
                "load_error_epochs": str(e_epochs),
            })

exploration = pd.DataFrame(rows)

print("Load failures:", (~exploration["load_ok"]).sum())
print("Data types observed:")
display(exploration.groupby("data_type").size().rename("n_subjects"))

exploration.head()


Load failures: 0
Data types observed:


data_type
epochs    10
Name: n_subjects, dtype: int64

,subject_id,file_path,data_type,sfreq,n_channels,duration_sec,n_epochs,epoch_len_sec,channel_hash,load_ok,load_error_raw
0,210,/Users/I743312/Documents/ketamine project/data...,epochs,250.0,62,112.0,14,8.0,ad0f5e8e1b7e,True,The number of trials is 14. It must be 1 for r...
1,219,/Users/I743312/Documents/ketamine project/data...,epochs,250.0,62,112.0,14,8.0,ad0f5e8e1b7e,True,The number of trials is 14. It must be 1 for r...
2,249,/Users/I743312/Documents/ketamine project/data...,epochs,250.0,62,120.0,15,8.0,ad0f5e8e1b7e,True,The number of trials is 15. It must be 1 for r...
3,251,/Users/I743312/Documents/ketamine project/data...,epochs,250.0,62,112.0,14,8.0,ad0f5e8e1b7e,True,The number of trials is 14. It must be 1 for r...
4,265,/Users/I743312/Documents/ketamine project/data...,epochs,250.0,62,104.0,13,8.0,ad0f5e8e1b7e,True,The number of trials is 13. It must be 1 for r...


In [15]:
# ------------------------------------------------
# 5.3 Sampling rate exploration
# ------------------------------------------------

print("Sampling rates observed:")
display(exploration.groupby("sfreq").size().rename("n_subjects"))

# Hard check
if exploration["sfreq"].nunique() > 1:
    print("WARNING: multiple sampling rates detected")

Sampling rates observed:


sfreq
250.0    10
Name: n_subjects, dtype: int64

In [16]:
# ------------------------------------------------
# 5.4 Channel consistency exploration
# ------------------------------------------------

print("Channel counts observed:")
display(exploration.groupby("n_channels").size().rename("n_subjects"))

print("Unique channel layouts (hashes):")
display(exploration.groupby("channel_hash").size().rename("n_subjects"))

if exploration["channel_hash"].nunique() > 1:
    print("WARNING: channel sets differ across subjects")

Channel counts observed:


n_channels
62    10
Name: n_subjects, dtype: int64

Unique channel layouts (hashes):


channel_hash
ad0f5e8e1b7e    10
Name: n_subjects, dtype: int64

In [17]:
# ------------------------------------------------
# 5.5 Recording duration exploration
# ------------------------------------------------

exploration["duration_min"] = exploration["duration_sec"] / 60.0

print("Recording duration (minutes):")
display(exploration["duration_min"].describe())

# Flag very short recordings
short_thresh_min = 2.0
n_short = (exploration["duration_min"] < short_thresh_min).sum()
print(f"Recordings shorter than {short_thresh_min} min:", n_short)

Recording duration (minutes):


count    10.000000
mean      1.880000
std       0.075686
min       1.733333
25%       1.866667
50%       1.866667
75%       1.866667
max       2.000000
Name: duration_min, dtype: float64

Recordings shorter than 2.0 min: 8


In [18]:
# ------------------------------------------------
# 5.6 Window-count simulation (no feature computation)
# ------------------------------------------------

WINDOW_LEN_SEC = 10.0
OVERLAP_FRAC = 0.5
STEP_SEC = WINDOW_LEN_SEC * (1 - OVERLAP_FRAC)

def estimate_n_windows(duration_sec, window_len, step_sec):
    if duration_sec < window_len:
        return 0
    return int(np.floor((duration_sec - window_len) / step_sec) + 1)

exploration["n_windows_10s"] = exploration["duration_sec"].apply(
    lambda d: estimate_n_windows(d, WINDOW_LEN_SEC, STEP_SEC)
)

print("Estimated number of 10s windows per recording:")
display(exploration["n_windows_10s"].describe())

print("Subjects with < 10 windows:",
      (exploration["n_windows_10s"] < 10).sum())

Estimated number of 10s windows per recording:


count    10.000000
mean     21.200000
std       1.135292
min      19.000000
25%      21.000000
50%      21.000000
75%      21.000000
max      23.000000
Name: n_windows_10s, dtype: float64

Subjects with < 10 windows: 0


In [18]:
sum(exploration.n_epochs)/len(exploration.n_epochs)

14.1